In [0]:
import random
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn import set_config
set_config(transform_output='pandas')

from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import PredictionErrorDisplay
from sklearn.metrics import PredictionErrorDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error
from sklearn.tree import export_text
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [0]:
df = spark.read.table("my_catalog.bsheffield_spark.nhs_combined_dataset").toPandas()

In [0]:
# -----------------------------
# CLEAN PREPROCESSING FUNCTIONS
# -----------------------------

def datetime_change(df):
    df = df.copy()
    df['closing_date'] = pd.to_datetime(df['closing_date'], errors='coerce')
    return df


def calc_qp_per_unit(df):
    df = df.copy()
    df['qc_per_unit'] = df['quoted_cost'] / df['quantity']
    return df


def bin_delivery_target(row):
    return 1 if row['quoted_cost'] == row['actual_cost'] else 0


def assign_month(df):
    df = df.copy()
    df['closing_date_month'] = df['closing_date'].dt.month_name()
    return df


def assign_season(month):
    if month in ("March", "April", "May"):
        return "Spring"
    elif month in ("June", "July", "August"):
        return "Summer"
    elif month in ("September", "October", "November"):
        return "Autumn"
    else:
        return "Winter"


def calc_qc_per_day(df):
    df = df.copy()
    df['qc_per_day'] = df['quoted_cost'] / df['quoted_lead_days']
    return df


# -----------------------------
# TRAIN/TEST SPLIT
# -----------------------------

def seperate_train_test(df):
    df = df.copy()

    # FEATURES (no target here)
    cat_feat = ['contract_type', 'postcode', 'closing_date_season', 'closing_date_month']
    num_feat = ['quantity', 'qc_per_unit', 'quoted_cost', 'qc_per_day']

    X = df[cat_feat + num_feat]
    y = df['bin_delivery_target']

    # EXTRA COLUMNS YOU NEED LATER
    extra = df[['contract_id','quoted_lead_days', 'closing_date_month', 'contract_type', 'quoted_cost', 'actual_cost', 'actual_lead_days']]

    X_train, X_test, y_train, y_test, extra_train, extra_test = train_test_split(
        X, y, extra, test_size=0.2, random_state=27
    )

    return X_train, X_test, y_train, y_test, extra_test



# -----------------------------
# TARGET ENCODING (LEAKAGE-FREE)
# -----------------------------

def target_encode_cols(X_train, X_test, y_train):
    X_train = X_train.copy()
    X_test = X_test.copy()

    # Season encoding
    season_means = y_train.groupby(X_train['closing_date_season']).mean()
    X_train['season_encoded'] = X_train['closing_date_season'].map(season_means)
    X_test['season_encoded'] = X_test['closing_date_season'].map(season_means)

    # Contract type encoding
    contract_means = y_train.groupby(X_train['contract_type']).mean()
    X_train['contract_encoded'] = X_train['contract_type'].map(contract_means)
    X_test['contract_encoded'] = X_test['contract_type'].map(contract_means)

    # Postcode encoding
    postcode_means = y_train.groupby(X_train['postcode']).mean()
    X_train['postcode_encoded'] = X_train['postcode'].map(postcode_means)
    X_test['postcode_encoded'] = X_test['postcode'].map(postcode_means)

    # Month encoding
    month_means = y_train.groupby(X_train['closing_date_month']).mean()
    X_train['month_encoded'] = X_train['closing_date_month'].map(month_means)
    X_test['month_encoded'] = X_test['closing_date_month'].map(month_means)

    # Drop raw columns (target not included)
    cols_to_drop = ['postcode', 'contract_type', 'closing_date_month']
    X_train = X_train.drop(columns=cols_to_drop)
    X_test = X_test.drop(columns=cols_to_drop)

    return X_train, X_test, season_means, contract_means, postcode_means, month_means


# -----------------------------
# IMPUTE + SCALE
# -----------------------------

def impute_scale_feats(X_train, X_test):
    cat_feat = ['closing_date_season']
    num_feat = [
        'quantity', 'qc_per_unit', 'quoted_cost', 'qc_per_day',
        'season_encoded', 'contract_encoded', 'postcode_encoded', 'month_encoded'
    ]

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    numerical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", MinMaxScaler())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", categorical_transformer, cat_feat),
            ("num", numerical_transformer, num_feat)
        ]
    )

    X_train_encoded = preprocessor.fit_transform(X_train)
    X_test_encoded = preprocessor.transform(X_test)

    return X_train_encoded, X_test_encoded, preprocessor


# -----------------------------
# FULL PREPROCESSING PIPELINE
# -----------------------------

def pp(df):
    df = df.copy()

    df = datetime_change(df)
    df = calc_qp_per_unit(df)
    df = assign_month(df)
    df['closing_date_season'] = df['closing_date_month'].apply(assign_season)
    df['bin_delivery_target'] = df.apply(bin_delivery_target, axis=1)
    df = calc_qc_per_day(df)

    X_train, X_test, y_train, y_test, extra_test = seperate_train_test(df)
    X_train, X_test, season_means, contract_means, postcode_means, month_means = target_encode_cols(X_train, X_test, y_train)
    X_train_encoded, X_test_encoded, preprocessor = impute_scale_feats(X_train, X_test)

    return X_train_encoded, X_test_encoded, y_train, y_test, extra_test, season_means, contract_means, postcode_means, month_means, preprocessor



# -----------------------------
# MODEL TRAINING
# -----------------------------

def lin_class_model(X_train_encoded, X_test_encoded, y_train, y_test, extra_test, season_means, contract_means, postcode_means, month_means, preprocessor):
    cost_model = LogisticRegression(class_weight='balanced')
    cost_model.fit(X_train_encoded, y_train)

    y_preds = cost_model.predict(X_test_encoded)
    y_pred_proba = cost_model.predict_proba(X_test_encoded)[:, 1]

    accuracy = accuracy_score(y_test, y_preds)
    precision = precision_score(y_test, y_preds)
    recall = recall_score(y_test, y_preds)
    f1 = f1_score(y_test, y_preds)

    lin_class_results_df_cost = pd.DataFrame({
        "Contract_ID": extra_test['contract_id'].values,
        "contract_type": extra_test['contract_type'].values,
        "Predicted": y_preds,
        "Actual": y_test.values,
        "Predicted_Probability": y_pred_proba,
        "quoted_lead_days": extra_test['quoted_lead_days'].values,
        "closing_date_month": extra_test['closing_date_month'].values,
        "quoted_cost": extra_test['quoted_cost'].values,
        "actual_cost": extra_test['actual_cost'].values})

    overall_lin_test_results_df_cost = pd.DataFrame([{
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }])

    return lin_class_results_df_cost, overall_lin_test_results_df_cost, cost_model, extra_test, season_means, contract_means, postcode_means, month_means, preprocessor


def lin_pipeline(df):
    X_train_encoded, X_test_encoded, y_train, y_test, extra_test, season_means, contract_means, postcode_means, month_means, preprocessor = pp(df)
    return lin_class_model(X_train_encoded, X_test_encoded, y_train, y_test, extra_test, season_means, contract_means, postcode_means, month_means, preprocessor)


In [0]:
lin_class_results_df_cost, overall_lin_test_results_df_cost, cost_model, extra_test, season_means, contract_means, postcode_means, month_means, preprocessor = lin_pipeline(df)

In [0]:
# spark.sql("CREATE VOLUME my_catalog.bsheffield_spark.model_store")


DataFrame[]

In [0]:
import pickle
import os, pickle

base = "/Volumes/my_catalog/bsheffield_spark/model_store/COST"
os.makedirs(base, exist_ok=True)

artefacts = {
    "season_means.pkl": season_means,
    "contract_means.pkl": contract_means,
    "postcode_means.pkl": postcode_means,
    "month_means.pkl": month_means,
    "preprocessor.pkl": preprocessor,
    "model.pkl": cost_model
}

for name, obj in artefacts.items():
    with open(f"{base}/{name}", "wb") as f:
        pickle.dump(obj, f)



In [0]:
spark.sql("DROP TABLE IF EXISTS my_catalog.bsheffield_spark.COST_lin_class_results_df")


In [0]:
# Save as Spark table for downstream tasks
spark_COST_lin_class_results_df = spark.createDataFrame(lin_class_results_df_cost)
spark_COST_lin_class_results_df.write.mode("overwrite").saveAsTable("my_catalog.bsheffield_spark.COST_lin_class_results_df")